In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from helpers.retrieval import search_chunks_local
from helpers.mlflow_retriever import load_retriever_artifacts

In [0]:
spark = SparkSession.builder.getOrCreate()
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA med")

In [0]:
latest = (
  spark.table("workspace.med.retriever_registry")
       .orderBy(F.col("created_at").desc())
       .limit(1)
       .collect()
)
if not latest:
    raise RuntimeError("No retriever runs found in registry")

RUN_ID = latest[0]["train_run_id"]
print("Using RUN_ID:", RUN_ID)
vectorizer, knn, chunk_ids, metadata = load_retriever_artifacts(RUN_ID)

In [0]:
# load chunk data in same order used during training
chunks_df = spark.table("workspace.med.doc_chunks")
serve_cols = ["chunk_id", "doc_id", "chunk_text", "source", "category", "title"]
filtered_df = chunks_df.select(*serve_cols).where(F.col("chunk_id").isin(chunk_ids))
chunks_pdf = filtered_df.toPandas()

present = set(chunks_pdf["chunk_id"].tolist())
missing = [cid for cid in chunk_ids if cid not in present]

print("trained chunk_ids:", len(chunk_ids))
print("present in table:", len(present))
print("missing ids:", len(missing))
if len(missing) > 0:
    print("example missing:", missing[:10])

pos_map = {cid: i for i, cid in enumerate(chunk_ids)}
chunks_pdf = chunks_pdf[chunks_pdf["chunk_id"].isin(pos_map)].copy()
chunks_pdf["__pos"] = chunks_pdf["chunk_id"].map(pos_map)
chunks_pdf = chunks_pdf.sort_values("__pos").drop(columns=["__pos"]).reset_index(drop=True)

print("chunks_pdf rows after reorder:", len(chunks_pdf))

if len(chunks_pdf) != len(chunk_ids):
    raise RuntimeError(
        f"Mismatch: knn trained on {len(chunk_ids)} chunks but current table has {len(chunks_pdf)}. "
        "Retrieval indices will be invalid. Re-train retriever using the current doc_chunks, "
        "or load chunk_text from the exact snapshot used for training."
    )

In [0]:
# test queries
questions = [
    "what is ibuprofen used for",
    "can i take ibuprofen with food",
    "what are side effects of nsaids",
]

for q in questions:
    print("\nQUESTION:", q)
    results = search_chunks_local(
        q,
        vectorizer=vectorizer,
        knn=knn,
        chunks_pdf=chunks_pdf,
        top_k=5,
    )

    for r in results:
        print(r["rank"], r["title"], r["cosine_distance"])
        print(r["chunk_text_preview"])
        print("-" * 80)